# SGD Comparison: EFGP vs Exact GP

Plain gradient descent (no momentum, no Adam) to isolate whether the
exp parameterization causes trajectory differences vs GPyTorch's softplus.

In [ ]:
import sys, time
import torch
import numpy as np
import matplotlib.pyplot as plt
import gpytorch

from efgpnd import EFGPND
from vanilla_gp_sampling import sample_gp_fast
from kernels.squared_exponential import SquaredExponential

plt.rcParams['figure.dpi'] = 120
dtype = torch.float64
torch.manual_seed(42)

## Data and config

In [ ]:
true_ls = 0.1
true_var = 1.0
true_noise = 1.0

n = 500
d = 1

torch.manual_seed(42)
x = torch.rand(n, d, dtype=dtype)
y = sample_gp_fast(x, length_scale=true_ls, variance=true_var,
                   noise_variance=true_noise, num_samples=1).squeeze()

init_ls = 0.5
init_var = 2.0
init_noise = 0.5
max_iters = 500

# EFGP settings
EPSILON = 1e-8
J = 50
cg_tol = 1e-8
noise_floor = 1e-5

print(f'n={n}, true: ls={true_ls}, var={true_var}, noise={true_noise}')

## GPyTorch Exact GP + SGD

In [ ]:
class _ExactGPModel(gpytorch.models.ExactGP):
    def __init__(self, train_x, train_y, likelihood):
        super().__init__(train_x, train_y, likelihood)
        self.mean_module = gpytorch.means.ZeroMean()
        self.base_kernel = gpytorch.kernels.RBFKernel()
        self.covar_module = gpytorch.kernels.ScaleKernel(self.base_kernel)
    def forward(self, x):
        return gpytorch.distributions.MultivariateNormal(
            self.mean_module(x), self.covar_module(x))

gpy_lr = 0.1

lik = gpytorch.likelihoods.GaussianLikelihood().to(dtype=dtype)
gp_model = _ExactGPModel(x, y, lik).to(dtype=dtype)
with torch.no_grad():
    gp_model.base_kernel.lengthscale = init_ls
    gp_model.covar_module.outputscale = init_var
    lik.noise = init_noise
gp_model.train(); lik.train()
opt_gp = torch.optim.SGD(gp_model.parameters(), lr=gpy_lr)
mll = gpytorch.mlls.ExactMarginalLogLikelihood(lik, gp_model)

gp_hist = {'iter': [], 'ls': [], 'var': [], 'noise': []}
t0 = time.time()
for i in range(max_iters):
    opt_gp.zero_grad()
    loss = -mll(gp_model(x), y)
    loss.backward()
    opt_gp.step()
    gp_hist['iter'].append(i)
    gp_hist['ls'].append(float(gp_model.base_kernel.lengthscale.detach().mean()))
    gp_hist['var'].append(float(gp_model.covar_module.outputscale.detach()))
    gp_hist['noise'].append(float(lik.noise.detach()))
    if i % 50 == 0:
        print(f'iter {i:>3d}  ls={gp_hist["ls"][-1]:.4f}  var={gp_hist["var"][-1]:.4f}  noise={gp_hist["noise"][-1]:.4f}')
gp_time = time.time() - t0
print(f'Exact GP time: {gp_time:.2f}s')

## EFGP + SGD

GPyTorch's `ExactMarginalLogLikelihood` normalizes the loss by `1/n` (`res.div_(num_data)`),
so its gradients are `n`× smaller than EFGP's. We compensate with `efgp_lr = gpy_lr / n`.

The remaining ~1.3× difference comes from the exp vs softplus Jacobian
(`d(pos)/d(raw)` = `pos` for exp vs `sigmoid(raw)` for softplus), which is
parameter-value-dependent and small.

In [ ]:
efgp_lr = gpy_lr / n

kernel_e = SquaredExponential(dimension=d, init_lengthscale=init_ls, init_variance=init_var)
model_e = EFGPND(x, y, kernel=kernel_e, sigmasq=init_noise, eps=EPSILON, estimate_params=False)
opt_e = torch.optim.SGD(model_e.parameters(), lr=efgp_lr)

efgp_hist = {'iter': [], 'ls': [], 'var': [], 'noise': []}
t0 = time.time()
for i in range(max_iters):
    opt_e.zero_grad()
    model_e.compute_gradients(trace_samples=J, cg_tol=cg_tol, noise_floor=noise_floor)
    opt_e.step()
    efgp_hist['iter'].append(i)
    efgp_hist['ls'].append(model_e.kernel.get_hyper('lengthscale'))
    efgp_hist['var'].append(model_e.kernel.get_hyper('variance'))
    efgp_hist['noise'].append(model_e._gp_params.sig2.item())
    if i % 50 == 0:
        print(f'iter {i:>3d}  ls={efgp_hist["ls"][-1]:.4f}  var={efgp_hist["var"][-1]:.4f}  noise={efgp_hist["noise"][-1]:.4f}')
efgp_time = time.time() - t0
print(f'EFGP time: {efgp_time:.2f}s')

## Learning curves

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

specs = [
    ('Lengthscale', 'ls', true_ls),
    ('Variance', 'var', true_var),
    ('Noise', 'noise', true_noise),
]

for ax, (title, key, true_val) in zip(axes, specs):
    ax.plot(gp_hist['iter'], gp_hist[key], 'g-', lw=1.5,
            label=f'Exact GP ({gp_time:.1f}s)')
    ax.plot(efgp_hist['iter'], efgp_hist[key], 'b-', lw=1.5,
            label=f'EFGP ({efgp_time:.1f}s)')
    ax.axhline(true_val, color='gray', ls=':', lw=1.5, label='True')
    ax.set_xlabel('Iteration')
    ax.set_title(title)
    ax.legend(fontsize=9)

fig.suptitle(f'SGD (no momentum): Exact GP lr={gpy_lr}, EFGP lr={efgp_lr:.6f} (= gpy_lr/n)', fontsize=12)
fig.tight_layout()
plt.show()

## Scale up: n = 5000

In [ ]:
n_big = 5000
torch.manual_seed(42)
x_big = torch.rand(n_big, d, dtype=dtype)
y_big = sample_gp_fast(x_big, length_scale=true_ls, variance=true_var,
                       noise_variance=true_noise, num_samples=1).squeeze()
print(f'n={n_big}')

In [ ]:
# Exact GP + SGD (n=5000)
gpy_lr_big = 0.1

lik_big = gpytorch.likelihoods.GaussianLikelihood().to(dtype=dtype)
gp_big = _ExactGPModel(x_big, y_big, lik_big).to(dtype=dtype)
with torch.no_grad():
    gp_big.base_kernel.lengthscale = init_ls
    gp_big.covar_module.outputscale = init_var
    lik_big.noise = init_noise
gp_big.train(); lik_big.train()
opt_gp_big = torch.optim.SGD(gp_big.parameters(), lr=gpy_lr_big)
mll_big = gpytorch.mlls.ExactMarginalLogLikelihood(lik_big, gp_big)

gp_hist_big = {'iter': [], 'ls': [], 'var': [], 'noise': []}
t0 = time.time()
for i in range(max_iters):
    opt_gp_big.zero_grad()
    loss = -mll_big(gp_big(x_big), y_big)
    loss.backward()
    opt_gp_big.step()
    gp_hist_big['iter'].append(i)
    gp_hist_big['ls'].append(float(gp_big.base_kernel.lengthscale.detach().mean()))
    gp_hist_big['var'].append(float(gp_big.covar_module.outputscale.detach()))
    gp_hist_big['noise'].append(float(lik_big.noise.detach()))
    if i % 50 == 0:
        print(f'iter {i:>3d}  ls={gp_hist_big["ls"][-1]:.4f}  var={gp_hist_big["var"][-1]:.4f}  noise={gp_hist_big["noise"][-1]:.4f}')
gp_time_big = time.time() - t0
print(f'Exact GP time: {gp_time_big:.2f}s')

In [ ]:
# EFGP + SGD (n=5000)
efgp_lr_big = gpy_lr_big / n_big

kernel_e_big = SquaredExponential(dimension=d, init_lengthscale=init_ls, init_variance=init_var)
model_e_big = EFGPND(x_big, y_big, kernel=kernel_e_big, sigmasq=init_noise, eps=EPSILON, estimate_params=False)
opt_e_big = torch.optim.SGD(model_e_big.parameters(), lr=efgp_lr_big)

efgp_hist_big = {'iter': [], 'ls': [], 'var': [], 'noise': []}
t0 = time.time()
for i in range(max_iters):
    opt_e_big.zero_grad()
    model_e_big.compute_gradients(trace_samples=J, cg_tol=cg_tol, noise_floor=noise_floor)
    opt_e_big.step()
    efgp_hist_big['iter'].append(i)
    efgp_hist_big['ls'].append(model_e_big.kernel.get_hyper('lengthscale'))
    efgp_hist_big['var'].append(model_e_big.kernel.get_hyper('variance'))
    efgp_hist_big['noise'].append(model_e_big._gp_params.sig2.item())
    if i % 50 == 0:
        print(f'iter {i:>3d}  ls={efgp_hist_big["ls"][-1]:.4f}  var={efgp_hist_big["var"][-1]:.4f}  noise={efgp_hist_big["noise"][-1]:.4f}')
efgp_time_big = time.time() - t0
print(f'EFGP time: {efgp_time_big:.2f}s')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

for ax, (title, key, true_val) in zip(axes, specs):
    ax.plot(gp_hist_big['iter'], gp_hist_big[key], 'g-', lw=1.5,
            label=f'Exact GP ({gp_time_big:.1f}s)')
    ax.plot(efgp_hist_big['iter'], efgp_hist_big[key], 'b-', lw=1.5,
            label=f'EFGP ({efgp_time_big:.1f}s)')
    ax.axhline(true_val, color='gray', ls=':', lw=1.5, label='True')
    ax.set_xlabel('Iteration')
    ax.set_title(title)
    ax.legend(fontsize=9)

fig.suptitle(f'SGD n={n_big}: Exact GP lr={gpy_lr_big}, EFGP lr={efgp_lr_big:.6f}', fontsize=12)
fig.tight_layout()
plt.show()